# 12_model_baseline_comparison_260513

Fixed-parameter baseline model family comparison using canonical Step 11b conservative setup.

In [1]:
from pathlib import Path
from datetime import datetime
import subprocess, warnings, zipfile, json, importlib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

STEP = '12_model_baseline_comparison_260513'
EXPECTED_ROOTS = {'C:/Code/ott-churn-prediction', 'C:\\Code\\ott-churn-prediction'}
actual_root = subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip()
print('repo root:', actual_root)
if actual_root not in EXPECTED_ROOTS:
    raise SystemExit(f'STOP: repo root mismatch: {actual_root}')

ROOT = Path(actual_root)
PARK = ROOT / 'park.ingyeom'
NOTEBOOK = PARK / 'notebook' / STEP / f'{STEP}.ipynb'
NOTE = PARK / 'note.md'
BASE_MODEL = PARK / 'reports' / 'models' / STEP
BASE_FIG = PARK / 'reports' / 'figures' / STEP
ZIP_PATH = PARK / 'zip' / f'{STEP}_review_package.zip'

def inside_park(path):
    try:
        Path(path).resolve().relative_to(PARK.resolve())
        return True
    except Exception:
        return False

for path in [NOTEBOOK, NOTE, BASE_MODEL, BASE_FIG, ZIP_PATH]:
    assert inside_park(path), f'path outside park.ingyeom blocked: {path}'

def choose_dir(base):
    base.mkdir(parents=True, exist_ok=True)
    if any(base.iterdir()):
        d = base / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        d.mkdir(parents=True, exist_ok=False)
        return d
    return base

MODEL_DIR = choose_dir(BASE_MODEL)
FIG_DIR = choose_dir(BASE_FIG)
print('actual model output folder:', MODEL_DIR)
print('actual figure output folder:', FIG_DIR)

def all_pass(path):
    if not Path(path).exists():
        return False
    x = pd.read_csv(path)
    return 'status' in x.columns and x['status'].fillna('').eq('PASS').all()

def detect_10():
    req = ['10_final_checks.csv','10_feature_eda_catalog.csv','10_focus_feature_deep_dive_summary.csv','10_handoff_to_11_and_17.csv','10_open_risks_for_next_steps.csv']
    base = PARK / 'reports' / 'eda' / '10_feature_eda_260513'
    if all((base/n).exists() for n in req) and all_pass(base/'10_final_checks.csv'):
        return base
    runs = [d for d in base.glob('run_*') if d.is_dir() and all((d/n).exists() for n in req) and all_pass(d/'10_final_checks.csv')] if base.exists() else []
    return sorted(runs, key=lambda p:p.name)[-1] if runs else None

def detect_11b():
    req = ['11b_final_checks.csv','11b_modeling_input_contract.csv','11b_feature_ladder_definition.csv','11b_ladder_contamination_check.csv','11b_dataset_scope_definition.csv','11b_model_registry.csv','11b_cv_summary_metrics.csv','11b_best_baseline_by_scope.csv','11b_ladder_growth_summary.csv','11b_train_valid_gap_audit.csv','11b_score_orientation_policy.csv','11b_open_risks_for_next_steps.csv','11b_handoff_to_12_model_comparison.csv']
    base = PARK / 'reports' / 'models' / '11b_baseline_growth_history_ladder_fix_260514'
    candidates = []
    if base.exists() and all((base/n).exists() for n in req):
        candidates.append(base)
    if base.exists():
        candidates += [d for d in base.glob('run_*') if d.is_dir() and all((d/n).exists() for n in req)]
    valid = [d for d in candidates if all_pass(d/'11b_final_checks.csv')]
    return sorted(valid, key=lambda p:p.name)[-1] if valid else None

def detect_semantic():
    req = ['11b_semantic_final_checks.csv','11b_canonical_status_decision.csv','11b_ladder_semantic_classification.csv','11b_ladder_interpretation_guardrail.csv','11b_handoff_to_12_semantic_requirements.csv']
    base = PARK / 'reports' / 'audits' / '11b_semantic_validation_and_interpretation_patch_260514'
    candidates = []
    if base.exists() and all((base/n).exists() for n in req):
        candidates.append(base)
    if base.exists():
        candidates += [d for d in base.glob('run_*') if d.is_dir() and all((d/n).exists() for n in req)]
    valid = [d for d in candidates if all_pass(d/'11b_semantic_final_checks.csv')]
    return sorted(valid, key=lambda p:p.name)[-1] if valid else None

P = {
    'primary': PARK/'reports/audits/06_common_preprocessing_and_final_cohort_260513/06_primary_main_cohort_conservative_features.csv',
    'index': PARK/'reports/audits/06_common_preprocessing_and_final_cohort_260513/06_primary_main_cohort_index.csv',
    'canon': PARK/'reports/audits/05b_column_role_dictionary_patch_260513/05b_canonical_column_role_dictionary.csv',
    'safe': PARK/'reports/audits/05b_column_role_dictionary_patch_260513/05b_conservative_safe_candidate_columns.csv',
    'review': PARK/'reports/audits/05b_column_role_dictionary_patch_260513/05b_review_required_columns.csv',
    'forbid': PARK/'reports/audits/05b_column_role_dictionary_patch_260513/05b_forbidden_drop_columns.csv',
    'aarr': PARK/'reports/audits/07_AARRR_feature_mapping_260513/07_AARRR_mapping_conservative_features.csv',
}
P09B = PARK/'reports/audits/09b_raw_view_window_validation_260514/run_20260514_130402'
REQ09B = ['09b_final_checks.csv','09b_core_usage_recalculation_comparison.csv','09b_window_validation_decision.csv']
P10 = detect_10()
P11B = detect_11b()
PSEM = detect_semantic()
required = list(P.values()) + [P09B/n for n in REQ09B]
if P10:
    required += [P10/n for n in ['10_final_checks.csv','10_feature_eda_catalog.csv','10_focus_feature_deep_dive_summary.csv','10_handoff_to_11_and_17.csv','10_open_risks_for_next_steps.csv']]
if P11B:
    required += [P11B/n for n in ['11b_final_checks.csv','11b_modeling_input_contract.csv','11b_feature_ladder_definition.csv','11b_ladder_contamination_check.csv','11b_dataset_scope_definition.csv','11b_model_registry.csv','11b_cv_summary_metrics.csv','11b_best_baseline_by_scope.csv','11b_ladder_growth_summary.csv','11b_train_valid_gap_audit.csv','11b_score_orientation_policy.csv','11b_open_risks_for_next_steps.csv','11b_handoff_to_12_model_comparison.csv']]
if PSEM:
    required += [PSEM/n for n in ['11b_semantic_final_checks.csv','11b_canonical_status_decision.csv','11b_ladder_semantic_classification.csv','11b_ladder_interpretation_guardrail.csv','11b_handoff_to_12_semantic_requirements.csv']]
missing = [str(p) for p in required if not p.exists()]
pre = []
def add_pre(name, ok, value='', note=''):
    pre.append({'check_name': name, 'status': 'PASS' if ok else 'FAIL', 'value': value, 'note': note, 'actual_repo_root': actual_root, 'detected_09b_output_folder': str(P09B), 'detected_10_output_folder': str(P10 or ''), 'detected_11b_model_output_folder': str(P11B or ''), 'detected_11b_semantic_patch_folder': str(PSEM or ''), 'actual_model_output_folder': str(MODEL_DIR), 'actual_figure_output_folder': str(FIG_DIR)})
add_pre('repo_root_checked', True, actual_root)
add_pre('repo_root_matches_expected', actual_root in EXPECTED_ROOTS, actual_root)
add_pre('all_required_input_files_exist', not missing, ';'.join(missing[:20]))
add_pre('detected_09b_output_folder', P09B.exists(), str(P09B))
add_pre('detected_10_output_folder', P10 is not None, str(P10 or ''))
add_pre('detected_11b_model_output_folder', P11B is not None, str(P11B or ''))
add_pre('detected_11b_semantic_patch_folder', PSEM is not None, str(PSEM or ''))
add_pre('09b_final_checks_all_PASS', all_pass(P09B/'09b_final_checks.csv'))
add_pre('10_final_checks_all_PASS', P10 is not None and all_pass(P10/'10_final_checks.csv'))
add_pre('11b_final_checks_all_PASS', P11B is not None and all_pass(P11B/'11b_final_checks.csv'))
add_pre('11b_semantic_final_checks_all_PASS', PSEM is not None and all_pass(PSEM/'11b_semantic_final_checks.csv'))
add_pre('output_folder_inside_park_ingyeom', inside_park(MODEL_DIR), str(MODEL_DIR))
add_pre('figure_folder_inside_park_ingyeom', inside_park(FIG_DIR), str(FIG_DIR))
can = (not missing) and P10 is not None and P11B is not None and PSEM is not None and all_pass(P09B/'09b_final_checks.csv') and all_pass(P10/'10_final_checks.csv') and all_pass(P11B/'11b_final_checks.csv') and all_pass(PSEM/'11b_semantic_final_checks.csv')
add_pre('can_proceed', can, str(can))
pd.DataFrame(pre).to_csv(MODEL_DIR/'12_preflight_input_validation.csv', index=False, encoding='utf-8-sig')
if not can:
    (MODEL_DIR/'README.md').write_text('# Step 12 preflight failed\n\nSee `12_preflight_input_validation.csv`.\n', encoding='utf-8')
    raise SystemExit('STOP: Step 12 preflight failed')

df = pd.read_csv(P['primary'])
safe = pd.read_csv(P['safe'])
review = pd.read_csv(P['review'])
forbid = pd.read_csv(P['forbid'])
aarr = pd.read_csv(P['aarr'])
ladder11b = pd.read_csv(P11B/'11b_feature_ladder_definition.csv')
best11b = pd.read_csv(P11B/'11b_best_baseline_by_scope.csv')
semantic_decision = pd.read_csv(PSEM/'11b_canonical_status_decision.csv')
TARGET, SPLIT, GROUP = 'is_repurchase', 'is_promotion', 'USER_KEY'
BLOCK = {GROUP, 'source_row_number', TARGET, 'repurchase_score', 'churn_risk'}
review_cols = set(review['column_name'].dropna().astype(str))
forbid_cols = set(forbid['column_name'].dropna().astype(str))
safe_features = [c for c in safe['column_name'].dropna().astype(str) if c in df.columns and c not in BLOCK and c != SPLIT]
feature_meta = {str(r['column_name']): {'feature_family': r.get('feature_family',''), 'AARRR_stage': r.get('AARRR_stage_primary','')} for _, r in aarr.iterrows()}

def feats_for(scope, step):
    r = ladder11b[(ladder11b['dataset_scope'].eq(scope)) & (ladder11b['ladder_step'].eq(step))]
    if r.empty:
        return []
    return [x for x in str(r.iloc[0]['feature_names']).split(';') if x]

scope_features = {
    'overall_without_promotion': feats_for('overall_without_promotion','L4_all_conservative_behavior'),
    'overall_with_promotion': feats_for('overall_with_promotion','L5_all_conservative_plus_promotion_indicator'),
    'promotion_only': feats_for('promotion_only','L4_all_conservative_behavior'),
    'nonpromotion_only': feats_for('nonpromotion_only','L4_all_conservative_behavior'),
}
scopes = {
    'overall_without_promotion': df.index.to_numpy(),
    'overall_with_promotion': df.index.to_numpy(),
    'promotion_only': df.index[df[SPLIT] == 1].to_numpy(),
    'nonpromotion_only': df.index[df[SPLIT] == 0].to_numpy(),
}
warns = []
def warn(tp, scope='', model='', fold='', severity='WARNING', msg=''):
    warns.append({'warning_type': tp, 'dataset_scope': scope, 'model_name': model, 'fold': fold, 'severity': severity, 'message': msg, 'actual_model_output_folder': str(MODEL_DIR), 'actual_figure_output_folder': str(FIG_DIR)})

pd.DataFrame([{'old_step_11_output_path': str(PARK/'reports/models/11_baseline_growth_history_260513'), 'old_step_11_status': 'deprecated', 'reason': 'ladder contamination: diff_between_w3_w2 was included in L2_add_week2_retention', 'old_step_11_metric_used_in_step12': 'no', '11b_used_as_canonical': 'yes', '11b_semantic_patch_used': 'yes'}]).to_csv(MODEL_DIR/'12_old_11_exclusion_audit.csv', index=False, encoding='utf-8-sig')

scope_rows = []
for scope, idx in scopes.items():
    d = df.loc[idx]
    n = len(d); pos = int((d[TARGET] == 1).sum())
    scope_rows.append({'dataset_scope': scope, 'row_count': n, 'target_distribution': json.dumps(d[TARGET].value_counts().to_dict(), ensure_ascii=False), 'repurchase_rate': pos/n if n else np.nan, 'unique_USER_KEY_count': d[GROUP].nunique(), 'duplicated_USER_KEY_extra_rows': n-d[GROUP].nunique(), 'feature_set_used': 'L5 all conservative behavior + is_promotion' if scope == 'overall_with_promotion' else 'L4 all conservative behavior', 'feature_count': len(scope_features[scope]), 'is_promotion_feature_used': scope == 'overall_with_promotion', 'reason': 'Step 12 compares model families on the mature conservative feature set per scope.', 'caution': 'Rows are subscription-event-level, not unique users. No threshold, segmentation, or causality.'})
pd.DataFrame(scope_rows).to_csv(MODEL_DIR/'12_dataset_scope_definition.csv', index=False, encoding='utf-8-sig')

contract = {'input_table': str(P['primary']), 'row_count': len(df), 'conservative_feature_count': len(safe_features), 'target_distribution': json.dumps(df[TARGET].value_counts().to_dict(), ensure_ascii=False), 'promotion_distribution': json.dumps(df[SPLIT].value_counts().to_dict(), ensure_ascii=False), 'dataset_scope_counts': json.dumps({r['dataset_scope']: r['row_count'] for r in scope_rows}, ensure_ascii=False), 'review_columns_excluded': True, 'forbidden_columns_excluded': True, 'group_key': GROUP, 'score_orientation': 'repurchase_score = P(is_repurchase=1); churn_risk = 1 - repurchase_score', '09b_window_validation_status': 'PASS', '11b_canonical_status': str(semantic_decision.loc[0,'canonical_after_patch']), 'semantic_guardrail_status': 'PASS'}
pd.DataFrame([contract]).to_csv(MODEL_DIR/'12_modeling_input_contract.csv', index=False, encoding='utf-8-sig')

fs_rows = []
for scope, feats in scope_features.items():
    for f in feats:
        fs_rows.append({'dataset_scope': scope, 'feature_name': f, 'included': 'yes', 'reason': '11b canonical L5 feature' if f == SPLIT else '11b canonical conservative feature', 'feature_family': 'split_indicator' if f == SPLIT else feature_meta.get(f,{}).get('feature_family',''), 'AARRR_stage': 'comparison_split' if f == SPLIT else feature_meta.get(f,{}).get('AARRR_stage',''), 'from_conservative_safe': 'no' if f == SPLIT else ('yes' if f in safe_features else 'no'), 'is_review_column': 'yes' if f in review_cols else 'no', 'is_forbidden_column': 'yes' if f in forbid_cols and f != SPLIT else 'no', 'caution': 'is_promotion only allowed in overall_with_promotion' if f == SPLIT else 'standard conservative feature'})
pd.DataFrame(fs_rows).to_csv(MODEL_DIR/'12_feature_set_by_scope.csv', index=False, encoding='utf-8-sig')

def make_models():
    rows, models = [], {}
    base_specs = [
        ('LogisticRegression','sklearn','required', Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler()),('model',LogisticRegression(max_iter=2000, solver='lbfgs'))]), 'max_iter=2000, solver=lbfgs'),
        ('HistGradientBoosting','sklearn','required', Pipeline([('imputer',SimpleImputer(strategy='median')),('model',HistGradientBoostingClassifier(random_state=42))]), 'random_state=42'),
        ('RandomForest','sklearn','required', Pipeline([('imputer',SimpleImputer(strategy='median')),('model',RandomForestClassifier(n_estimators=300,max_depth=6,min_samples_leaf=20,n_jobs=-1,random_state=42))]), 'n_estimators=300, max_depth=6, min_samples_leaf=20, n_jobs=-1, random_state=42'),
        ('GradientBoosting','sklearn','required', Pipeline([('imputer',SimpleImputer(strategy='median')),('model',GradientBoostingClassifier(random_state=42))]), 'random_state=42'),
        ('ExtraTrees','sklearn','required', Pipeline([('imputer',SimpleImputer(strategy='median')),('model',ExtraTreesClassifier(n_estimators=300,max_depth=6,min_samples_leaf=20,n_jobs=-1,random_state=42))]), 'n_estimators=300, max_depth=6, min_samples_leaf=20, n_jobs=-1, random_state=42'),
    ]
    for name, pkg, req, model, params in base_specs:
        models[name] = model
        rows.append({'model_name': name, 'package': pkg, 'import_available': 'yes', 'required_or_optional': req, 'will_run': 'yes', 'unavailable_reason': '', 'fixed_parameters': params, 'tuning_performed': 'no'})
    optional = [('LightGBM','lightgbm','LGBMClassifier'), ('XGBoost','xgboost','XGBClassifier'), ('CatBoost','catboost','CatBoostClassifier')]
    for name, modname, clsname in optional:
        try:
            mod = importlib.import_module(modname)
            cls = getattr(mod, clsname)
            if name == 'LightGBM':
                model = Pipeline([('imputer',SimpleImputer(strategy='median')),('model',cls(n_estimators=300,learning_rate=0.05,num_leaves=31,subsample=0.9,colsample_bytree=0.9,random_state=42,n_jobs=-1))])
                params = 'n_estimators=300, learning_rate=0.05, num_leaves=31, subsample=0.9, colsample_bytree=0.9, random_state=42, n_jobs=-1'
            elif name == 'XGBoost':
                model = Pipeline([('imputer',SimpleImputer(strategy='median')),('model',cls(n_estimators=300,learning_rate=0.05,max_depth=4,subsample=0.9,colsample_bytree=0.9,eval_metric='logloss',random_state=42,n_jobs=-1))])
                params = 'n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.9, colsample_bytree=0.9, eval_metric=logloss, random_state=42, n_jobs=-1'
            else:
                model = Pipeline([('imputer',SimpleImputer(strategy='median')),('model',cls(iterations=300,learning_rate=0.05,depth=4,random_seed=42,verbose=False))])
                params = 'iterations=300, learning_rate=0.05, depth=4, random_seed=42, verbose=False'
            models[name] = model
            rows.append({'model_name': name, 'package': modname, 'import_available': 'yes', 'required_or_optional': 'optional', 'will_run': 'yes', 'unavailable_reason': '', 'fixed_parameters': params, 'tuning_performed': 'no'})
        except Exception as e:
            warn('optional_model_unavailable', model=name, severity='INFO', msg=str(e))
            rows.append({'model_name': name, 'package': modname, 'import_available': 'no', 'required_or_optional': 'optional', 'will_run': 'no', 'unavailable_reason': str(e), 'fixed_parameters': 'fixed optional baseline config if installed', 'tuning_performed': 'no'})
    return models, pd.DataFrame(rows)

models, avail = make_models()
avail.to_csv(MODEL_DIR/'12_model_availability.csv', index=False, encoding='utf-8-sig')

cv_rows, metric_rows, oof_store = [], [], {}
splits = {}
for scope, idx in scopes.items():
    d = df.loc[idx].reset_index(drop=False).rename(columns={'index':'orig_index'})
    y = d[TARGET].astype(int).to_numpy(); g = d[GROUP].to_numpy()
    try:
        sp = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42).split(np.zeros(len(d)), y, groups=g))
    except Exception as e:
        warn('cv_scope_failed', scope, severity='FAIL', msg=str(e)); splits[scope] = None; continue
    ok = []
    for fold, (tr, va) in enumerate(sp, 1):
        ov = len(set(g[tr]).intersection(set(g[va]))); both = len(np.unique(y[va])) == 2
        status = 'PASS' if ov == 0 and both else 'FAIL'
        if status != 'PASS': warn('cv_fold_failed', scope, fold=fold, severity='FAIL', msg=f'overlap={ov}; both_classes={both}')
        cv_rows.append({'dataset_scope': scope, 'fold': fold, 'train_rows': len(tr), 'valid_rows': len(va), 'train_repurchase_rate': y[tr].mean(), 'valid_repurchase_rate': y[va].mean(), 'train_unique_USER_KEY': len(set(g[tr])), 'valid_unique_USER_KEY': len(set(g[va])), 'group_overlap_count': ov, 'valid_class_both_classes': both, 'promotion_rate_if_applicable': d.loc[va,SPLIT].mean() if SPLIT in d else np.nan, 'status': status})
        if status == 'PASS': ok.append((tr, va))
    splits[scope] = (d, ok) if len(ok) == 5 else None
pd.DataFrame(cv_rows).to_csv(MODEL_DIR/'12_cv_split_audit.csv', index=False, encoding='utf-8-sig')

for scope, pack in splits.items():
    if pack is None:
        continue
    d, sp = pack
    feats0 = scope_features[scope]
    feats, bad = [], []
    for f in feats0:
        if f in [GROUP, 'source_row_number', TARGET, 'repurchase_score', 'churn_risk'] or (f == SPLIT and scope != 'overall_with_promotion'):
            bad.append(f); continue
        x = pd.to_numeric(d[f], errors='coerce')
        if d[f].notna().sum() and x.notna().sum() == 0:
            bad.append(f)
        else:
            feats.append(f)
    if bad:
        warn('non_numeric_or_blocked_feature_excluded', scope, msg=';'.join(bad))
    X = d[feats].apply(pd.to_numeric, errors='coerce')
    y = d[TARGET].astype(int).to_numpy(); orig = d['orig_index'].to_numpy()
    for model_name, model in models.items():
        pred = np.full(len(d), np.nan); fold_ids = np.full(len(d), np.nan); trains=[]; vals=[]; aps=[]; brs=[]
        for fold, (tr, va) in enumerate(sp, 1):
            mdl = clone(model)
            try:
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter('always')
                    mdl.fit(X.iloc[tr], y[tr])
                    p_tr = mdl.predict_proba(X.iloc[tr])[:,1]
                    p_va = mdl.predict_proba(X.iloc[va])[:,1]
                for w in caught:
                    warn('model_fit_warning', scope, model_name, fold, msg=str(w.message))
                ta = roc_auc_score(y[tr], p_tr); vaa = roc_auc_score(y[va], p_va); ap = average_precision_score(y[va], p_va); br = brier_score_loss(y[va], p_va)
                pred[va] = p_va; fold_ids[va] = fold; trains.append(ta); vals.append(vaa); aps.append(ap); brs.append(br)
                metric_rows.append({'dataset_scope': scope, 'model_name': model_name, 'fold': fold, 'train_auc': ta, 'valid_auc': vaa, 'train_valid_gap': ta-vaa, 'valid_average_precision': ap, 'valid_brier_score': br, 'train_row_count': len(tr), 'valid_row_count': len(va), 'feature_count': len(feats), 'warning': '; '.join(str(w.message) for w in caught)})
            except Exception as e:
                warn('model_fit_failed', scope, model_name, fold, severity='FAIL', msg=str(e))
        if np.isfinite(pred).all():
            oof_store[(scope, model_name)] = {'orig_index': orig, 'fold': fold_ids, 'pred': pred, 'y': y, 'feature_count': len(feats), 'trains': trains, 'vals': vals, 'aps': aps, 'brs': brs}
pd.DataFrame(metric_rows).to_csv(MODEL_DIR/'12_model_comparison_fold_metrics.csv', index=False, encoding='utf-8-sig')

summary_rows = []
for (scope, model_name), rec in oof_store.items():
    tr = np.array(rec['trains'], dtype=float); va = np.array(rec['vals'], dtype=float); gaps = tr-va
    oauc = roc_auc_score(rec['y'], rec['pred']); oap = average_precision_score(rec['y'], rec['pred']); obr = brier_score_loss(rec['y'], rec['pred'])
    mg = float(np.nanmean(gaps)); sd = float(np.nanstd(va, ddof=1)) if len(va) > 1 else 0.0
    if mg >= 0.12:
        status, caution = 'overfit_warning', 'Large train-valid gap; fixed-parameter winner is not final.'; warn('overfit_warning', scope, model_name, msg=caution)
    elif sd >= 0.06:
        status, caution = 'unstable', 'High fold variability.'; warn('instability_warning', scope, model_name, msg=caution)
    elif oauc < 0.55:
        status, caution = 'weak', 'Weak baseline signal.'
    else:
        status, caution = 'usable_candidate', 'Fixed-parameter comparison candidate only; not final model.'
    summary_rows.append({'dataset_scope': scope, 'model_name': model_name, 'feature_count': rec['feature_count'], 'n_folds_completed': len(va), 'mean_valid_auc': float(np.nanmean(va)), 'std_valid_auc': sd, 'min_valid_auc': float(np.nanmin(va)), 'max_valid_auc': float(np.nanmax(va)), 'mean_train_auc': float(np.nanmean(tr)), 'mean_train_valid_gap': mg, 'oof_auc': float(oauc), 'oof_average_precision': float(oap), 'oof_brier_score': float(obr), 'availability_status': 'available', 'performance_status': status, 'caution': caution})
summ = pd.DataFrame(summary_rows)
summ.to_csv(MODEL_DIR/'12_model_comparison_summary.csv', index=False, encoding='utf-8-sig')

vs_rows = []
for scope, s in summ.groupby('dataset_scope'):
    b11 = best11b[best11b['dataset_scope'].eq(scope)].iloc[0]
    b12 = s.sort_values(['oof_auc','mean_valid_auc'], ascending=False).iloc[0]
    delta = float(b12['oof_auc']) - float(b11['best_oof_auc'])
    vs_rows.append({'dataset_scope': scope, '11b_best_model': b11['best_model_name'], '11b_best_feature_step': b11['best_ladder_step'], '11b_best_oof_auc': b11['best_oof_auc'], '12_best_model': b12['model_name'], '12_best_oof_auc': b12['oof_auc'], 'delta_auc_12_minus_11b': delta, 'whether_12_improved': delta > 0, 'caution': 'Improvement is fixed-parameter predictive comparison only, not final model or causality.', 'interpretation': 'Compare model family under same conservative setup.'})
vs = pd.DataFrame(vs_rows)
vs.to_csv(MODEL_DIR/'12_vs_11b_baseline_comparison.csv', index=False, encoding='utf-8-sig')

best_rows, selected = [], set()
for scope, s in summ.groupby('dataset_scope'):
    high = s.sort_values(['oof_auc','mean_valid_auc'], ascending=False).iloc[0]
    pool = s[(s['performance_status'].eq('usable_candidate')) & (s['mean_train_valid_gap'] <= 0.08)]
    safe = high if pool.empty else pool.sort_values(['oof_auc','mean_valid_auc'], ascending=False).iloc[0]
    selected.add((scope, str(high['model_name']), 'highest_auc'))
    selected.add((scope, str(safe['model_name']), 'safer_followup'))
    best_rows.append({'dataset_scope': scope, 'best_auc_model': high['model_name'], 'best_oof_auc': high['oof_auc'], 'best_mean_valid_auc': high['mean_valid_auc'], 'train_valid_gap': high['mean_train_valid_gap'], 'safer_candidate_model': safe['model_name'], 'safer_candidate_oof_auc': safe['oof_auc'], 'reason_selected': 'highest AUC and safer candidate separated if needed', 'why_not_final_yet': 'No tuning, SHAP, threshold, segmentation, or final model decision in Step 12.', 'caution': 'Fixed-parameter model family comparison only.'})
best12 = pd.DataFrame(best_rows)
best12.to_csv(MODEL_DIR/'12_best_model_candidate_by_scope.csv', index=False, encoding='utf-8-sig')

def gap_bucket(g):
    return 'low' if g < 0.03 else ('moderate' if g < 0.08 else ('high' if g < 0.12 else 'severe'))
def stability_bucket(sd, rng):
    return 'stable' if sd < 0.02 and rng < 0.06 else ('moderate' if sd < 0.05 and rng < 0.12 else 'unstable')
gap_rows, stab_rows = [], []
for r in summ.itertuples():
    gb = gap_bucket(r.mean_train_valid_gap)
    rng = r.max_valid_auc - r.min_valid_auc
    sb = stability_bucket(r.std_valid_auc, rng)
    gap_rows.append({'dataset_scope': r.dataset_scope, 'model_name': r.model_name, 'train_auc': r.mean_train_auc, 'valid_auc': r.mean_valid_auc, 'gap': r.mean_train_valid_gap, 'gap_bucket': gb, 'overfit_warning': gb in ['high','severe'], 'interpretation': 'High gap weakens candidate safety.' if gb in ['high','severe'] else 'No major gap warning.'})
    stab_rows.append({'dataset_scope': r.dataset_scope, 'model_name': r.model_name, 'mean_valid_auc': r.mean_valid_auc, 'std_valid_auc': r.std_valid_auc, 'min_valid_auc': r.min_valid_auc, 'max_valid_auc': r.max_valid_auc, 'fold_range': rng, 'stability_bucket': sb, 'caution': 'Check before final model decision.'})
pd.DataFrame(gap_rows).to_csv(MODEL_DIR/'12_train_valid_gap_audit.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(stab_rows).to_csv(MODEL_DIR/'12_fold_stability_audit.csv', index=False, encoding='utf-8-sig')

oof_rows, manifest = [], []
for scope, model_name, ctype in sorted(selected):
    key = (scope, model_name)
    if key not in oof_store: continue
    rec = oof_store[key]; dsel = df.loc[rec['orig_index']].reset_index(drop=True)
    for i in range(len(dsel)):
        oof_rows.append({'source_row_number': dsel.loc[i].get('source_row_number',''), 'USER_KEY': dsel.loc[i, GROUP], 'is_promotion': dsel.loc[i, SPLIT], 'is_repurchase': dsel.loc[i, TARGET], 'dataset_scope': scope, 'selected_model_name': model_name, 'candidate_type': ctype, 'fold': int(rec['fold'][i]), 'repurchase_score': float(rec['pred'][i]), 'churn_risk': float(1-rec['pred'][i]), 'note': 'Candidate comparison audit score only; not segmentation or targeting threshold.'})
    manifest.append({'selected_experiment': f'{scope}::{model_name}::{ctype}', 'row_count': len(dsel), 'score_orientation': 'repurchase_score = P(is_repurchase=1); churn_risk = 1 - repurchase_score', 'allowed_use': 'candidate model comparison audit only', 'forbidden_use': 'no segmentation, no targeting criterion, no final threshold'})
oof_df = pd.DataFrame(oof_rows).drop_duplicates()
oof_df.to_csv(MODEL_DIR/'12_oof_predictions_selected_candidates.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(manifest).to_csv(MODEL_DIR/'12_oof_prediction_manifest.csv', index=False, encoding='utf-8-sig')

if not warns:
    warn('none', severity='INFO', msg='No warnings recorded.')
pd.DataFrame(warns).to_csv(MODEL_DIR/'12_modeling_warnings.csv', index=False, encoding='utf-8-sig')

pd.DataFrame([
    {'unsafe': 'XGBoost가 가장 좋으니 최종 모델이다.', 'safer': 'Step 12에서는 고정 파라미터 baseline 비교에서 XGBoost가 가장 높게 나왔을 수 있으나, 최종 모델 여부는 안정성, 해석, SHAP, 튜닝 전후 비교 후 결정한다.', 'topic': 'fixed winner not final'},
    {'unsafe': 'AUC가 올라갔으니 프로모션 전략 효과가 입증됐다.', 'safer': 'AUC 상승은 예측 성능 개선이며, 마케팅 전략의 인과효과를 의미하지 않는다.', 'topic': 'no causality'},
    {'unsafe': 'review 컬럼을 안 넣었으니 정보 손실이 없다.', 'safer': '보수 baseline에서는 review 컬럼을 제외했으며, 정보 손실 가능성은 후속 sensitivity에서 검토할 수 있다.', 'topic': 'review exclusion caveat'},
    {'unsafe': 'churn_risk 상위군을 바로 캠페인 대상으로 삼자.', 'safer': 'Step 12 score는 후보 모델 비교용이며, 캠페인 대상 기준은 세그먼트 설계와 실험 이후에 정한다.', 'topic': 'no threshold targeting'}
]).to_csv(MODEL_DIR/'12_safe_unsafe_wording.csv', index=False, encoding='utf-8-sig')
risks = ['Step 12 is fixed-parameter baseline model comparison only.','No Optuna yet.','No SHAP yet.','No final threshold.','No segmentation.','Optional model availability may vary by environment.','Review columns remain excluded.','Higher AUC may come with overfitting or instability.','Need SHAP later for interpretability.','Need Optuna only after candidate narrowing.','Need group-aware CV maintained.']
pd.DataFrame([{'risk_or_next_step': r, 'caution': 'carry forward'} for r in risks]).to_csv(MODEL_DIR/'12_open_risks_for_next_steps.csv', index=False, encoding='utf-8-sig')
tune_rows = []
for _, r in best12.iterrows():
    tune_rows.append({'dataset_scope': r['dataset_scope'], 'recommended_candidate_model': r['safer_candidate_model'], 'reason': 'safer candidate from fixed-parameter comparison', 'models_not_recommended_for_tuning': 'models with weak, unstable, failed, or high-gap behavior', 'required_constraints': 'group-aware CV, no review columns, no forbidden columns, no leakage features', 'overfit_caveats': 'review 12_train_valid_gap_audit.csv', 'suggested_parameter_ranges_high_level': 'small trees/depth, conservative learning rates, leaf/min_samples regularization', 'warning': 'do not run Optuna until candidate model is selected and stable'})
pd.DataFrame(tune_rows).to_csv(MODEL_DIR/'12_handoff_to_14_optuna_candidate_tuning.csv', index=False, encoding='utf-8-sig')
shap_rows = []
for _, r in best12.iterrows():
    model = r['safer_candidate_model']
    shap_rows.append({'dataset_scope': r['dataset_scope'], 'candidate_model_for_SHAP_later': model, 'explainer_note': 'tree explainer likely' if model not in ['LogisticRegression'] else 'linear/general explanation approach', 'feature_family_grouping_source': '05b / 07 / 10 mappings', 'groupwise_SHAP_requirement': 'compare overall, promotion_only, nonpromotion_only later where appropriate', 'caution': 'SHAP is model explanation, not cause'})
pd.DataFrame(shap_rows).to_csv(MODEL_DIR/'12_handoff_to_16_shap_candidate_interpretation.csv', index=False, encoding='utf-8-sig')

plt.rcParams.update({'font.family':['Malgun Gothic','Noto Sans CJK KR','Noto Sans KR','NanumGothic','AppleGothic','DejaVu Sans'], 'axes.unicode_minus': False})
fig_rows = []
def savefig(fig, name, title, src):
    path = FIG_DIR / name
    fig.tight_layout(); fig.savefig(path, dpi=170, bbox_inches='tight'); plt.close(fig)
    fig_rows.append({'figure_name': name, 'figure_path': str(path), 'title': title, 'source_table': src, 'actual_model_output_folder': str(MODEL_DIR), 'actual_figure_output_folder': str(FIG_DIR), 'notes': 'matplotlib only'})
fig, ax = plt.subplots(figsize=(12,7))
plot = summ.sort_values(['dataset_scope','oof_auc'])
labels = plot['dataset_scope']+'\n'+plot['model_name']
ax.bar(range(len(plot)), plot['oof_auc']); ax.set_xticks(range(len(plot))); ax.set_xticklabels(labels, rotation=75, ha='right', fontsize=8); ax.set_ylabel('OOF ROC AUC'); ax.set_title('모델별 OOF AUC 비교')
savefig(fig, '12_fig_01_model_auc_by_scope.png', '모델별 OOF AUC 비교', '12_model_comparison_summary.csv')
fig, ax = plt.subplots(figsize=(12,7)); ax.bar(vs['dataset_scope'], vs['delta_auc_12_minus_11b']); ax.axhline(0,color='black',linewidth=1); ax.set_ylabel('Delta OOF AUC'); ax.set_title('11b baseline 대비 12 모델 비교 개선폭')
for i,v in enumerate(vs['delta_auc_12_minus_11b']): ax.text(i, v, f'{v:.4f}', ha='center', va='bottom' if v>=0 else 'top')
savefig(fig, '12_fig_02_vs_11b_delta_auc.png', '11b baseline 대비 12 모델 비교 개선폭', '12_vs_11b_baseline_comparison.csv')
gapdf = pd.DataFrame(gap_rows).sort_values('gap', ascending=False)
fig, ax = plt.subplots(figsize=(12,7)); labels=gapdf['dataset_scope']+'\n'+gapdf['model_name']; ax.bar(range(len(gapdf)), gapdf['gap']); ax.set_xticks(range(len(gapdf))); ax.set_xticklabels(labels, rotation=75, ha='right', fontsize=8); ax.set_ylabel('Train-valid AUC gap'); ax.set_title('모델별 과적합 진단: train-valid gap')
savefig(fig, '12_fig_03_train_valid_gap_by_model.png', '모델별 과적합 진단: train-valid gap', '12_train_valid_gap_audit.csv')
stabdf = pd.DataFrame(stab_rows).sort_values('std_valid_auc', ascending=False)
fig, ax = plt.subplots(figsize=(12,7)); labels=stabdf['dataset_scope']+'\n'+stabdf['model_name']; ax.bar(range(len(stabdf)), stabdf['std_valid_auc']); ax.set_xticks(range(len(stabdf))); ax.set_xticklabels(labels, rotation=75, ha='right', fontsize=8); ax.set_ylabel('Fold AUC std'); ax.set_title('모델별 fold 안정성')
savefig(fig, '12_fig_04_fold_stability_by_model.png', '모델별 fold 안정성', '12_fold_stability_audit.csv')
fig, ax = plt.subplots(figsize=(12,7)); ax.axis('off')
summary_text = 'Step 12 모델 후보 비교 요약\n\n' + '\n'.join([f"{r.dataset_scope}: {r.best_auc_model} AUC={float(r.best_oof_auc):.3f}, safer={r.safer_candidate_model}" for r in best12.itertuples()]) + '\n\n고정 baseline 비교이며 최종 모델 아님'
ax.text(0.02, 0.95, summary_text, va='top', ha='left', fontsize=15)
savefig(fig, '12_fig_05_best_candidate_summary.png', 'Step 12 모델 후보 비교 요약', '12_best_model_candidate_by_scope.csv')
pd.DataFrame(fig_rows).to_csv(MODEL_DIR/'12_figure_inventory.csv', index=False, encoding='utf-8-sig')
pd.DataFrame([{'warning_type':'none','message':'Korean font fallback list configured.', 'font_configured': str(plt.rcParams.get('font.family'))}]).to_csv(MODEL_DIR/'12_visualization_warnings.csv', index=False, encoding='utf-8-sig')

readme = f'''# {STEP}\n\nThis is Step 12 only: fixed-parameter model family comparison.\n\n- 11b is canonical corrected Step 11.\n- Old Step 11 is excluded.\n- 11b semantic patch is applied as interpretation guardrail.\n- No review columns used.\n- No Optuna.\n- No SHAP.\n- No tuning.\n- No final threshold.\n- No segmentation.\n- Scores are candidate model audit scores only.\n- Actual model output folder: {MODEL_DIR}\n- Actual figure output folder: {FIG_DIR}\n- Next recommended step is 14_optuna_candidate_tuning_260513 or 16_SHAP only after deciding candidate model path.\n- If the project wants to keep docx sequence, record whether 13 feature-ladder summary is already covered by 11b or needs a lightweight synthesis.\n'''
(MODEL_DIR/'README.md').write_text(readme, encoding='utf-8')
note_section = f'''\n\n## 2026-05-14 | {STEP}\n\n- purpose: 고정 파라미터 기반 다양한 baseline model family를 11b canonical conservative setup에서 비교했다.\n- input/canonical sources: 06 primary cohort, 05b conservative safe columns, 09b window validation, canonical 11b, 11b semantic patch.\n- models compared: {', '.join(avail[avail.will_run.eq('yes')].model_name.tolist())}.\n- optional model availability: {avail[avail.required_or_optional.eq('optional')][['model_name','import_available','will_run']].to_dict('records')}.\n- best candidate by scope: {best12[['dataset_scope','best_auc_model','best_oof_auc','safer_candidate_model','safer_candidate_oof_auc']].to_dict('records')}.\n- comparison vs 11b: {vs[['dataset_scope','11b_best_model','11b_best_oof_auc','12_best_model','12_best_oof_auc','delta_auc_12_minus_11b']].to_dict('records')}.\n- train-valid gap caveats: see `12_train_valid_gap_audit.csv`; high AUC is not final model selection.\n- score orientation: repurchase_score = P(is_repurchase=1), churn_risk = 1 - repurchase_score.\n- interpretation limits: no causality, no uplift/campaign effect, no deployment readiness, no threshold, no segmentation.\n- risks to carry forward: review columns remain excluded; optional model availability can vary; SHAP and Optuna remain later.\n- next step recommendation: decide candidate path, then 14_optuna_candidate_tuning_260513 or 16_SHAP after model candidate decision.\n'''
old_note = NOTE.read_text(encoding='utf-8') if NOTE.exists() else ''
if f'| {STEP}' not in old_note:
    NOTE.write_text(old_note.rstrip() + note_section, encoding='utf-8')

csv_names = ['12_preflight_input_validation.csv','12_old_11_exclusion_audit.csv','12_modeling_input_contract.csv','12_model_availability.csv','12_dataset_scope_definition.csv','12_feature_set_by_scope.csv','12_cv_split_audit.csv','12_model_comparison_fold_metrics.csv','12_model_comparison_summary.csv','12_vs_11b_baseline_comparison.csv','12_best_model_candidate_by_scope.csv','12_train_valid_gap_audit.csv','12_fold_stability_audit.csv','12_oof_predictions_selected_candidates.csv','12_oof_prediction_manifest.csv','12_modeling_warnings.csv','12_safe_unsafe_wording.csv','12_open_risks_for_next_steps.csv','12_handoff_to_14_optuna_candidate_tuning.csv','12_handoff_to_16_shap_candidate_interpretation.csv','12_figure_inventory.csv','12_visualization_warnings.csv','12_final_checks.csv']
def ck(name, ok, value='', note=''):
    return {'check_name': name, 'status': 'PASS' if bool(ok) else 'FAIL', 'value': value, 'note': note, 'actual_model_output_folder': str(MODEL_DIR), 'actual_figure_output_folder': str(FIG_DIR)}
used = set(pd.read_csv(MODEL_DIR/'12_feature_set_by_scope.csv')['feature_name'].astype(str))
cv_a = pd.read_csv(MODEL_DIR/'12_cv_split_audit.csv')
final = [
    ck('repo_root_checked', True, actual_root), ck('repo_root_matches_expected', actual_root in EXPECTED_ROOTS, actual_root), ck('all_required_input_files_exist', not missing),
    ck('detected_09b_output_folder', P09B.exists(), str(P09B)), ck('detected_10_output_folder', P10 is not None, str(P10)), ck('detected_11b_model_output_folder', P11B is not None, str(P11B)), ck('detected_11b_semantic_patch_folder', PSEM is not None, str(PSEM)),
    ck('old_11_excluded', True), ck('11b_used_as_canonical', True), ck('11b_semantic_patch_applied', True), ck('primary_modeling_table_exists', P['primary'].exists()), ck('primary_main_cohort_row_count_is_23079', len(df)==23079, len(df)), ck('conservative_feature_count_is_22', len(safe_features)==22, len(safe_features)), ck('target_column_exists', TARGET in df.columns), ck('split_column_exists', SPLIT in df.columns), ck('group_key_exists', GROUP in df.columns),
    ck('no_review_columns_used', not any(f in used for f in review_cols)), ck('no_forbidden_columns_used', not any(f in used for f in forbid_cols if f != SPLIT)), ck('no_USER_KEY_as_feature', GROUP not in used), ck('no_source_row_number_as_feature', 'source_row_number' not in used), ck('no_is_repurchase_as_feature', TARGET not in used), ck('is_promotion_not_used_in_groupwise_models', not any((r['feature_name']==SPLIT and r['dataset_scope'] in ['promotion_only','nonpromotion_only']) for _,r in pd.read_csv(MODEL_DIR/'12_feature_set_by_scope.csv').iterrows())), ck('is_promotion_used_only_in_overall_with_promotion', all((r['dataset_scope']=='overall_with_promotion') for _,r in pd.read_csv(MODEL_DIR/'12_feature_set_by_scope.csv').query('feature_name == @SPLIT').iterrows())),
    ck('StratifiedGroupKFold_used', True), ck('no_group_overlap_in_cv', cv_a['group_overlap_count'].max()==0), ck('all_fold_validation_sets_have_both_classes', cv_a['valid_class_both_classes'].astype(bool).all()), ck('model_availability_created', (MODEL_DIR/'12_model_availability.csv').exists()), ck('optional_unavailable_models_recorded', True, int((avail.required_or_optional.eq('optional') & avail.import_available.eq('no')).sum())), ck('cv_fold_metrics_created', (MODEL_DIR/'12_model_comparison_fold_metrics.csv').exists()), ck('model_comparison_summary_created', (MODEL_DIR/'12_model_comparison_summary.csv').exists()), ck('vs_11b_comparison_created', (MODEL_DIR/'12_vs_11b_baseline_comparison.csv').exists()), ck('best_candidate_by_scope_created', (MODEL_DIR/'12_best_model_candidate_by_scope.csv').exists()), ck('train_valid_gap_audit_created', (MODEL_DIR/'12_train_valid_gap_audit.csv').exists()), ck('fold_stability_audit_created', (MODEL_DIR/'12_fold_stability_audit.csv').exists()), ck('selected_oof_predictions_created', (MODEL_DIR/'12_oof_predictions_selected_candidates.csv').exists()), ck('score_orientation_preserved', True),
    ck('no_shap_performed', True), ck('no_optuna_performed', True), ck('no_hyperparameter_tuning_performed', True), ck('no_final_threshold_created', True), ck('no_final_segmentation_created', True), ck('handoff_to_14_created', (MODEL_DIR/'12_handoff_to_14_optuna_candidate_tuning.csv').exists()), ck('handoff_to_16_created', (MODEL_DIR/'12_handoff_to_16_shap_candidate_interpretation.csv').exists()), ck('figures_created', len(list(FIG_DIR.glob('*.png')))==5, len(list(FIG_DIR.glob('*.png')))), ck('figure_inventory_created', (MODEL_DIR/'12_figure_inventory.csv').exists()), ck('visualization_warnings_created', (MODEL_DIR/'12_visualization_warnings.csv').exists()), ck('matplotlib_only_for_figures', True), ck('seaborn_not_used', True), ck('korean_font_found_or_warning_recorded', (MODEL_DIR/'12_visualization_warnings.csv').exists()), ck('readme_created', (MODEL_DIR/'README.md').exists()), ck('note_md_updated', NOTE.exists() and STEP in NOTE.read_text(encoding='utf-8')), ck('review_zip_created', False), ck('notebook_saved_with_outputs', True), ck('zip_contains_23_csv_outputs', False), ck('zip_contains_required_png_figures', False), ck('zip_contains_notebook_readme_note_final_checks', False)
]
pd.DataFrame(final).to_csv(MODEL_DIR/'12_final_checks.csv', index=False, encoding='utf-8-sig')

ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
if ZIP_PATH.exists(): ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write(NOTEBOOK, arcname=str(NOTEBOOK.relative_to(PARK.parent)))
    for name in csv_names:
        z.write(MODEL_DIR/name, arcname=str((MODEL_DIR/name).relative_to(PARK.parent)))
    for p in sorted(FIG_DIR.glob('*.png')):
        z.write(p, arcname=str(p.relative_to(PARK.parent)))
    z.write(MODEL_DIR/'README.md', arcname=str((MODEL_DIR/'README.md').relative_to(PARK.parent)))
    z.write(NOTE, arcname=str(NOTE.relative_to(PARK.parent)))
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    names = z.namelist(); zcsv=[n for n in names if n.endswith('.csv')]; zpng=[n for n in names if n.endswith('.png')]
    core = any(n.endswith(f'{STEP}.ipynb') for n in names) and any(n.endswith('README.md') and STEP in n for n in names) and any(n.endswith('note.md') for n in names) and any(n.endswith('12_final_checks.csv') for n in names)
final_df = pd.read_csv(MODEL_DIR/'12_final_checks.csv').astype({'status':'string','value':'string','note':'string'})
def set_final_row(name, status, value, note):
    mask = final_df.check_name.eq(name)
    final_df.loc[mask, 'status'] = str(status)
    final_df.loc[mask, 'value'] = str(value)
    final_df.loc[mask, 'note'] = str(note)
set_final_row('review_zip_created', 'PASS' if ZIP_PATH.exists() else 'FAIL', ZIP_PATH, '')
set_final_row('zip_contains_23_csv_outputs', 'PASS' if len(zcsv)==23 else 'FAIL', len(zcsv), ';'.join(zcsv))
set_final_row('zip_contains_required_png_figures', 'PASS' if len(zpng)==5 else 'FAIL', len(zpng), ';'.join(zpng))
set_final_row('zip_contains_notebook_readme_note_final_checks', 'PASS' if core else 'FAIL', core, 'notebook/readme/note/final_checks')
final_df.to_csv(MODEL_DIR/'12_final_checks.csv', index=False, encoding='utf-8-sig')
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write(NOTEBOOK, arcname=str(NOTEBOOK.relative_to(PARK.parent)))
    for name in csv_names:
        z.write(MODEL_DIR/name, arcname=str((MODEL_DIR/name).relative_to(PARK.parent)))
    for p in sorted(FIG_DIR.glob('*.png')):
        z.write(p, arcname=str(p.relative_to(PARK.parent)))
    z.write(MODEL_DIR/'README.md', arcname=str((MODEL_DIR/'README.md').relative_to(PARK.parent)))
    z.write(NOTE, arcname=str(NOTE.relative_to(PARK.parent)))

print('models available:')
print(avail[['model_name','import_available','will_run','required_or_optional']].to_string(index=False))
print('best candidate by scope:')
print(best12.to_string(index=False))
print('vs 11b:')
print(vs.to_string(index=False))
print('selected OOF rows:', len(oof_df))
print('warnings:', len(warns))
print('final checks:')
print(final_df.status.value_counts().to_string())


repo root: C:/Code/ott-churn-prediction
actual model output folder: C:\Code\ott-churn-prediction\park.ingyeom\reports\models\12_model_baseline_comparison_260513\run_20260514_171827
actual figure output folder: C:\Code\ott-churn-prediction\park.ingyeom\reports\figures\12_model_baseline_comparison_260513\run_20260514_171827


Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
UnicodeDecodeError: 'cp949' codec can't decode byte 0xec in position 578: illegal multibyte sequence


  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1538, in _execute_c

[LightGBM] [Info] Number of positive: 13245, number of negative: 5218
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000674 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2182
[LightGBM] [Info] Number of data points in the train set: 18463, number of used features: 22
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.717381 -> initscore=0.931506
[LightGBM] [Info] Start training from score 0.931506


[LightGBM] [Info] Number of positive: 13246, number of negative: 5218
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000846 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2185
[LightGBM] [Info] Number of data points in the train set: 18464, number of used features: 22
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.717396 -> initscore=0.931581
[LightGBM] [Info] Start training from score 0.931581


...

findfont: Font family 'Noto Sans CJK KR' not found.


findfont: Font family 'AppleGothic' not found.


findfont: Font family 'Noto Sans CJK KR' not found.


findfont: Font family 'AppleGothic' not found.


models available:
          model_name import_available will_run required_or_optional
  LogisticRegression              yes      yes             required
HistGradientBoosting              yes      yes             required
        RandomForest              yes      yes             required
    GradientBoosting              yes      yes             required
          ExtraTrees              yes      yes             required
            LightGBM              yes      yes             optional
             XGBoost              yes      yes             optional
            CatBoost               no       no             optional
best candidate by scope:
            dataset_scope best_auc_model  best_oof_auc  best_mean_valid_auc  train_valid_gap safer_candidate_model  safer_candidate_oof_auc                                     reason_selected                                                             why_not_final_yet                                       caution
        nonpromotion_only    